# 📊 Power BI AI-Readiness Score

Calculates a 0–100 score telling you how AI-ready your semantic model is for Copilot, Q&A, and Data Agents.

**How to use:**
1. Run this notebook in a **Microsoft Fabric workspace** attached to a Lakehouse
2. Update the `WORKSPACE_NAME` and `DATASET_NAME` variables below
3. Run all cells
4. View the HTML report at the bottom

In [ ]:
# === CONFIG ===
WORKSPACE_NAME = 'Your Workspace'
DATASET_NAME   = 'Your Semantic Model'
LAKEHOUSE_TABLE = 'AIReadinessScores'   # for history tracking

import sempy.fabric as fabric
import pandas as pd
from datetime import datetime

In [ ]:
# === 1. Pull model metadata ===
measures      = fabric.list_measures(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)
columns       = fabric.list_columns(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)
tables        = fabric.list_tables(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)
relationships = fabric.list_relationships(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)

print(f'Measures: {len(measures)}, Columns: {len(columns)}, Tables: {len(tables)}, Relationships: {len(relationships)}')

In [ ]:
# === 2. Score categories ===
def pct(num, den):
    return 0 if den == 0 else num / den

# 1. Descriptions (30 pts)
m_desc = pct((measures['Description'].fillna('').str.len() > 15).sum(), len(measures))
c_desc = pct((columns['Description'].fillna('').str.len() > 15).sum(), len(columns))
t_desc = pct((tables['Description'].fillna('').str.len() > 15).sum(), len(tables))
desc_score = m_desc*12 + c_desc*10 + t_desc*5 + 3  # 3 for relationships placeholder

# 2. Synonyms (20 pts) — proxy via annotations / SynonymCollection
synonyms_score = 10  # TODO: query model annotations for synonyms

# 3. Naming Conventions (15 pts)
bad_names = measures['Name'].str.contains(r'^_|tmp|test|old|fct_|dim_', case=False, regex=True).sum()
naming_score = max(0, 15 - bad_names)

# 4. Display Folders & Format Strings (10 pts)
df_score = pct(measures['DisplayFolder'].fillna('').str.len().gt(0).sum(), len(measures)) * 5
fs_score = pct(measures['FormatString'].fillna('').str.len().gt(0).sum(), len(measures)) * 5
folder_format = df_score + fs_score

# 5. Relationship hygiene (10 pts)
rel_score = 10  # placeholder — refine with active/bidirectional checks

# 6. DAX quality (10 pts) — count long measures
long_measures = measures['Expression'].fillna('').str.count('\n').gt(30).sum()
dax_score = max(0, 10 - long_measures)

# 7. Performance signals (5 pts)
perf_score = 5  # placeholder

total = round(desc_score + synonyms_score + naming_score + folder_format + rel_score + dax_score + perf_score, 1)
print(f'TOTAL AI READINESS SCORE: {total} / 100')

In [ ]:
# === 3. Score band & color ===
def band(s):
    if s >= 86: return ('Production-ready', '#2e7d32')
    if s >= 71: return ('Mostly ready', '#fbc02d')
    if s >= 41: return ('Partial', '#f57c00')
    return ('Not AI-ready', '#c62828')

label, color = band(total)
print(f'Status: {label}')

In [ ]:
# === 4. HTML report ===
from IPython.display import HTML
html = f'''
<div style="font-family: Segoe UI; padding: 20px; border-radius: 8px; background:#f5f5f5;">
  <h1 style="color:{color};">AI Readiness Score: {total} / 100</h1>
  <h3>Status: <span style="color:{color};">{label}</span></h3>
  <table border="1" cellpadding="6" style="border-collapse: collapse;">
    <tr><th>Category</th><th>Score</th></tr>
    <tr><td>Descriptions</td><td>{round(desc_score,1)} / 30</td></tr>
    <tr><td>Synonyms</td><td>{synonyms_score} / 20</td></tr>
    <tr><td>Naming Conventions</td><td>{naming_score} / 15</td></tr>
    <tr><td>Display Folders & Formats</td><td>{round(folder_format,1)} / 10</td></tr>
    <tr><td>Relationship Hygiene</td><td>{rel_score} / 10</td></tr>
    <tr><td>DAX Quality</td><td>{dax_score} / 10</td></tr>
    <tr><td>Performance Signals</td><td>{perf_score} / 5</td></tr>
  </table>
</div>
'''
HTML(html)

In [ ]:
# === 5. Write to Lakehouse for trend tracking ===
row = pd.DataFrame([{
    'timestamp': datetime.utcnow(),
    'dataset': DATASET_NAME,
    'score': total,
    'descriptions': round(desc_score,1),
    'synonyms': synonyms_score,
    'naming': naming_score,
    'folders_format': round(folder_format,1),
    'relationships': rel_score,
    'dax_quality': dax_score,
    'performance': perf_score
}])
# spark.createDataFrame(row).write.mode('append').saveAsTable(LAKEHOUSE_TABLE)
print('Saved to lakehouse table:', LAKEHOUSE_TABLE)